# DSHN: what directionality actually buys on TopoBench liftings

Supplementary analysis for the TDL Challenge 2026 submission of
**Directional Sheaf Hypergraph Networks** (Mule et al., ICLR 2026,
[arXiv:2510.04727](https://arxiv.org/abs/2510.04727)).

The official `run_evaluation.ipynb` benchmarks the model on GraphUniverse. This
notebook answers a question that benchmark structurally cannot: **DSHN's
headline contribution is a complex-valued *directed* sheaf Laplacian, but every
TopoBench graph-to-hypergraph lifting produces an *undirected* hypergraph.**

We establish three things:

1. On undirected input the operator is **exactly real** and its spectrum is
   **exactly independent of the charge $q$**. So $q$ has no effect there whatsoever.
2. The paper's own Appendix D.5 orientation, applied to a node-centred
   lifting, produces a genuinely directed hypergraph and switches the complex
   machinery on. We expose this as `orientation: star`.
3. What DSHN gives on undirected data is positive semidefiniteness. Its
   diagonal coefficient $1 - 1/\delta_e$, against $1/\delta_e$ in Duta et al.
   (2023), is the entire difference, and it is what makes the operator a valid
   diffusion operator.

Point 3 matters for this challenge specifically: PR #321 implements
SheafHyperGNN, which *is* the $1/\delta_e$ operator, so points 1-3 map the
boundary between the two submissions.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch

from topobench.nn.backbones.hypergraph.dshn import DSHN
from topobench.nn.backbones.hypergraph.dshn_utils.laplacian import (
    charge_phase,
    derive_orientation,
    directed_sheaf_laplacian,
)

torch.manual_seed(0)
DTYPE = torch.float64  # the claims below are exact, so measure them in float64


def star_hypergraph(n, stride=3):
    '''Node-centred hypergraph: hyperedge j is the neighbourhood of node j.

    This is the shape a k-hop lifting produces (one hyperedge per node), which
    is what `orientation="star"` requires.
    '''
    pairs = []
    for v in range(n):
        pairs.append((v, v))
        pairs.extend(
            (u, v) for u in range(n) if u != v and (u + v) % stride == 0
        )
    return torch.tensor(pairs, dtype=torch.long).T


def laplacian(edge_index, n, m, d, q, is_head, normalized=True):
    '''Dense Laplacian for a fixed random sheaf.'''
    g = torch.Generator().manual_seed(42)
    blocks = torch.randn(
        edge_index.size(1), d, d, generator=g, dtype=DTYPE
    )
    phase = charge_phase(is_head, q, dtype=torch.cdouble)
    return directed_sheaf_laplacian(
        edge_index, blocks, phase, n, m, normalized=normalized
    ).to_dense()

## 1. Undirected input: the charge $q$ does nothing at all

With every incidence in the tail set ($H(e) = \emptyset$, the paper's
definition of an undirected hyperedge, after Gallo et al. 1993), every phase
product of Eq. 4 is

$$\overline{S^{(q)}_{u \lhd e}} S^{(q)}_{v \lhd e}
  = \overline{e^{-2\pi i q}} e^{-2\pi i q} = 1,$$

independently of $q$. The paper states the consequence on p. 5 -- $L^F$ is
"real-valued if the hypergraph is undirected" -- and Theorem 6 identifies the
result as the undirected hypergraph Laplacian of Zhou et al. (2006).

We sweep $q$ and measure the imaginary mass and the spectrum.

In [ ]:
N, D = 24, 3
ei = star_hypergraph(N)
M = N
undirected = derive_orientation(ei, N, M, "none")

rows = []
reference = None
for q in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.37]:
    q_n = laplacian(ei, N, M, D, q, undirected)
    lap = torch.eye(N * D, dtype=torch.cdouble) - q_n
    spectrum = torch.linalg.eigvalsh(lap)
    if reference is None:
        reference = spectrum
    rows.append(
        {
            "q": q,
            "max|Im(L_N)|": lap.imag.abs().max().item(),
            "lambda_min": spectrum.min().item(),
            "lambda_max": spectrum.max().item(),
            "spectrum drift vs q=0": (spectrum - reference).abs().max().item(),
        }
    )

print(f"{'q':>6} {'max|Im|':>12} {'lambda_min':>12} {'lambda_max':>12} {'drift':>12}")
for r in rows:
    print(
        f"{r['q']:>6.2f} {r['max|Im(L_N)|']:>12.2e} {r['lambda_min']:>12.6f} "
        f"{r['lambda_max']:>12.6f} {r['spectrum drift vs q=0']:>12.2e}"
    )

The imaginary part is identically zero and the spectrum does not move. Note
also $\lambda_{\max} = 1$ exactly, so Theorem 3's bound is tight, and
$\lambda_{\min} \geq 0$, which is Corollary 1.

What this means for the challenge benchmark: run on GraphUniverse as lifted,
DSHN and DSHNLight are real-valued models and the `q` entry in their configs
does nothing.

## 2. Inducing an orientation switches the complex part on

Appendix D.5 orients a hyperedge away from the node it is centred on:
$T(e_v) = \{v\}$, $H(e_v) = N(v)$. The paper applies this to directed source
graphs; applied to an undirected one it still yields a genuinely directed
hypergraph, because the *centre* of each hyperedge is distinguished.

TopoBench's default `graph2hypergraph` lifting is `khop`, which builds exactly
one hyperedge per node, so the centre is recoverable and the construction
applies directly.

In [ ]:
directed = derive_orientation(ei, N, M, "star")
print(f"head-set fraction: {directed.float().mean():.3f}")

rows = []
for q in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]:
    lap_u = torch.eye(N * D, dtype=torch.cdouble) - laplacian(
        ei, N, M, D, q, undirected
    )
    lap_d = torch.eye(N * D, dtype=torch.cdouble) - laplacian(
        ei, N, M, D, q, directed
    )
    ev_d = torch.linalg.eigvalsh(lap_d)
    rows.append((q, lap_u.imag.abs().max().item(),
                 lap_d.imag.abs().max().item(),
                 ev_d.min().item(), ev_d.max().item()))

print(f"\n{'q':>6} {'undirected |Im|':>17} {'star |Im|':>12} {'l_min':>10} {'l_max':>10}")
for q, iu, idr, lo, hi in rows:
    print(f"{q:>6.2f} {iu:>17.2e} {idr:>12.4f} {lo:>10.6f} {hi:>10.6f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
qs = [r[0] for r in rows]
ax.plot(qs, [r[1] for r in rows], "o--", label="orientation='none'")
ax.plot(qs, [r[2] for r in rows], "s-", label="orientation='star'")
ax.set_xlabel("charge $q$")
ax.set_ylabel(r"$\max|\Im(L_N^F)|$")
ax.set_title("Directionality is inert until an orientation is induced")
ax.legend()
fig.tight_layout()
plt.show()

With the star orientation the imaginary part grows with $q$ and the spectrum
shifts, while the operator stays Hermitian and inside $[0, 1]$ -- the theorems
hold in the directed case too, which is the point of the construction.

## 3. What the undirected case does give: positive semidefiniteness

Appendix E gives the counterexample. On $V=\{0,1,2,3\}$,
$E=\{\{0,1,2\},\{1,2,3\}\}$ with a trivial sheaf, the Duta et al. (2023)
operator has eigenvalues $\{4/3, 1/3, (1\pm\sqrt{17})/6\}$, so
$\lambda_{\min} = (1-\sqrt{17})/6 < 0$ and it is not a valid diffusion
operator. The two operators share every off-diagonal block and differ only in
the diagonal coefficient.

In [ ]:
APPENDIX_E = torch.tensor(
    [(0, 0), (1, 0), (2, 0), (1, 1), (2, 1), (3, 1)], dtype=torch.long
).T
trivial = torch.eye(1, dtype=DTYPE).expand(6, 1, 1).contiguous()
phase = charge_phase(torch.zeros(6, dtype=torch.bool), 0.0, dtype=torch.cdouble)

dshn_lap = directed_sheaf_laplacian(
    APPENDIX_E, trivial, phase, 4, 2, normalized=False
).to_dense().real

# Same off-diagonals, 1/delta_e on the diagonal instead of 1 - 1/delta_e.
off = dshn_lap - torch.diag(torch.diagonal(dshn_lap))
duta_lap = off + torch.diag(
    torch.tensor([1 / 3, 2 / 3, 2 / 3, 1 / 3], dtype=DTYPE)
)

print("DSHN  L^F diagonal:", torch.diagonal(dshn_lap).tolist())
print("Duta  L^F diagonal:", torch.diagonal(duta_lap).tolist())
print()
print("DSHN  eigenvalues:", [round(v, 6) for v in torch.linalg.eigvalsh(dshn_lap).tolist()])
print("Duta  eigenvalues:", [round(v, 6) for v in torch.linalg.eigvalsh(duta_lap).tolist()])
print()
print(f"paper's closed form (1-sqrt(17))/6 = {(1 - math.sqrt(17)) / 6:.6f}")
print(f"DSHN is PSD: {torch.linalg.eigvalsh(dshn_lap).min() >= -1e-12}")
print(f"Duta is PSD: {torch.linalg.eigvalsh(duta_lap).min() >= -1e-12}")

The gap is not an artefact of one hand-picked hypergraph. We sample random
hypergraphs with random sheaves and compare $\lambda_{\min}$ of the two
normalized operators.

In [ ]:
def random_hypergraph(n, m, lo, hi, gen):
    pairs = []
    for e in range(m):
        size = int(torch.randint(lo, hi, (1,), generator=gen))
        for v in torch.randperm(n, generator=gen)[:size].tolist():
            pairs.append((v, e))
    covered = {v for v, _ in pairs}
    for v in range(n):
        if v not in covered:
            pairs.append((v, int(torch.randint(m, (1,), generator=gen))))
    return torch.tensor(pairs, dtype=torch.long).T


gen = torch.Generator().manual_seed(7)
dshn_mins, duta_mins = [], []
for _ in range(200):
    n, m, d = 16, 9, 2
    e_idx = random_hypergraph(n, m, 3, 7, gen)
    blocks = torch.randn(e_idx.size(1), d, d, generator=gen, dtype=DTYPE)
    ph = charge_phase(torch.zeros(e_idx.size(1), dtype=torch.bool), 0.0,
                      dtype=torch.cdouble)

    q_n = directed_sheaf_laplacian(e_idx, blocks, ph, n, m).to_dense()
    lap = torch.eye(n * d, dtype=torch.cdouble) - q_n
    dshn_mins.append(torch.linalg.eigvalsh(lap).min().item())

    # The Duta diagonal: swap sum_e (1 - 1/delta) F^T F for sum_e (1/delta) F^T F.
    raw = directed_sheaf_laplacian(
        e_idx, blocks, ph, n, m, normalized=False
    ).to_dense().real
    delta = torch.bincount(e_idx[1], minlength=m)[e_idx[1]].to(DTYPE)
    prod = torch.bmm(blocks.transpose(1, 2), blocks)
    swap = torch.zeros(n, d, d, dtype=DTYPE)
    swap.index_add_(
        0, e_idx[0], (2.0 / delta - 1.0).view(-1, 1, 1) * prod
    )
    duta_raw = raw + torch.block_diag(*torch.unbind(swap, 0))
    duta_mins.append(torch.linalg.eigvalsh(duta_raw).min().item())

dshn_mins, duta_mins = np.array(dshn_mins), np.array(duta_mins)
print(f"DSHN  (normalized L_N): non-PSD in {(dshn_mins < -1e-9).sum():3d}/200 "
      f"trials, worst lambda_min = {dshn_mins.min():+.6f}")
print(f"Duta  (unnormalized L): non-PSD in {(duta_mins < -1e-9).sum():3d}/200 "
      f"trials, worst lambda_min = {duta_mins.min():+.6f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(duta_mins, bins=40, alpha=0.75, label=r"Duta et al.: $1/\delta_e$")
ax.axvline(0.0, color="k", lw=1, ls="--")
ax.set_xlabel(r"$\lambda_{\min}$")
ax.set_ylabel("count")
ax.set_title(r"The $1/\delta_e$ diagonal admits negative eigenvalues")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Does an induced orientation change what the model learns?

Everything above is an exact statement about the operator. Whether an *induced*
orientation helps a downstream task is a separate, empirical question, and one
the paper does not address, since it only ever orients genuinely directed data.

We build a directed 3-level hierarchy, where edges only ever point from level
$k$ to level $k+1$, and lift it as $e_v = \{v\} \cup N_{out}(v)$ -- the
Appendix D.5 construction. Node features are noise, so the label is recoverable
only from structure, and we hold out half the nodes, because this backbone will
happily memorise a graph this small and tell us nothing.

The incidence structure $e_v = \{v\} \cup N_{out}(v)$
is *already* asymmetric, so `orientation="none"` is not direction-blind here.
What `star` adds is a phase distinguishing each hyperedge's centre from its
members. The comparison below isolates that phase, not directionality at large.

In [ ]:
def hierarchy_task(n=240, levels=3, p_forward=0.06, seed=0):
    """Directed level graph, lifted as e_v = {v} U N_out(v).

    The label is a node's level, recoverable from edge direction but not from
    features, which are pure noise.
    """
    g = torch.Generator().manual_seed(seed)
    level = torch.arange(n) * levels // n
    pairs = [(v, v) for v in range(n)]
    for u in range(n):
        forward = (level == level[u] + 1).nonzero(as_tuple=True)[0]
        keep = torch.rand(forward.numel(), generator=g) < p_forward
        # v in N_out(u) means v belongs to the hyperedge centred on u.
        pairs.extend((int(v), u) for v in forward[keep].tolist())

    e_idx = torch.tensor(pairs, dtype=torch.long).T
    inc = torch.sparse_coo_tensor(
        e_idx, torch.ones(e_idx.size(1)), (n, n)
    ).coalesce()

    x = torch.randn(n, 8, generator=g)
    perm = torch.randperm(n, generator=g)
    train_mask = torch.zeros(n, dtype=torch.bool)
    train_mask[perm[: n // 2]] = True
    return x, inc, level, train_mask, ~train_mask


def run_one(orientation, q, steps=250, seed=0):
    """Train on half the nodes, report accuracy on the held-out half."""
    torch.manual_seed(seed)
    x, inc, y, tr, te = hierarchy_task(seed=seed)
    model = DSHN(
        8, 32, int(y.max()) + 1, n_layers=2, d=2,
        orientation=orientation, q=q, dropout=0.5,
    )
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(steps):
        opt.zero_grad()
        out = model(x, inc)[0]
        torch.nn.functional.cross_entropy(out[tr], y[tr]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        out = model(x, inc)[0]
    return (out[te].argmax(1) == y[te]).float().mean().item()


SEEDS = range(5)
header = ("orientation", "q", "held-out accuracy")
print(f"{header[0]:>12} {header[1]:>6} {header[2]:>20}")
for orientation, q in [("none", 0.00), ("none", 0.25),
                       ("star", 0.00), ("star", 0.10),
                       ("star", 0.20), ("star", 0.25)]:
    accs = np.array([run_one(orientation, q, seed=s) for s in SEEDS])
    print(
        f"{orientation:>12} {q:>6.2f}        "
        f"{accs.mean():.3f} +/- {accs.std():.3f}"
    )

Three of those rows should be identical, and they are:

- `none` at $q=0$ vs $q=0.25$ -- the operator does not depend on $q$ on
  undirected input, which is §1's invariance now visible end to end through
  training rather than at the operator level.
- `star` at $q=0$ matches both -- at $q=0$ the charge is $1$ everywhere, so an
  orientation exists but carries no phase, and the operator is the undirected
  one again. The paper makes this explicit on p. 5: "By setting $q = 0$, the
  hypergraph directions are entirely disregarded."

The movement happens only when an orientation *and* a non-zero charge are both
present. That is the one place in this pipeline where DSHN's complex machinery
can affect an outcome, and it is unreachable from TopoBench without
`orientation="star"`.

One small synthetic task with five seeds, and a task built so that direction is
the signal: treat the magnitudes as indicative and the invariances as exact.

## Summary

| Claim | Where | Status |
|---|---|---|
| Undirected input gives a real operator, spectrum independent of $q$ | §1, Eq. 4 / Thm 6 | exact, verified |
| $\lambda_{\max}(L_N^F) = 1$, $\lambda_{\min} \geq 0$ | §1, Thm 3 / Cor 1 | exact, verified |
| Appendix D.5 orientation activates the complex terms | §2 | verified |
| $1 - 1/\delta_e$ makes the operator PSD where $1/\delta_e$ does not | §3, App. E | verified, matches closed form |

The takeaway for the submission: on GraphUniverse as lifted, DSHN is a
real-valued model whose contribution over SheafHyperGNN is a well-posed
diffusion operator, not directionality. `orientation="star"` is what makes the
directed half of the paper reachable from TopoBench, and it is shipped as
`configs/model/hypergraph/dshn_directed.yaml`.